## Step 1

In [1]:
# Install the Google ADK
!pip install google-adk -q

## Step 2

In [2]:
def get_lat_lng(location_name: str) -> dict:
    """
    Converts a city or location name into latitude and longitude using Google Maps API.

    Args:
        location_name: The name of the city or location (e.g., 'Austin, TX').

    Returns:
        A dictionary containing 'lat' and 'lng'.
    """
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location_name}&key={GOOGLE_MAPS_API_KEY}"
    response = requests.get(url).json()

    if response["status"] == "OK":
        geometry = response["results"][0]["geometry"]["location"]
        return {"lat": geometry["lat"], "lng": geometry["lng"]}
    else:
        return {"error": "Location not found."}

def get_nws_weather(lat: float, lng: float) -> str:
    """
    Retrieves real-time weather alerts and conditions from the National Weather Service.

    Args:
        lat: The latitude of the location.
        lng: The longitude of the location.

    Returns:
        A string summary of the forecast or an error message.
    """
    # NWS requires a User-Agent header
    headers = {'User-Agent': '(myweatherapp.com, contact@example.com)'}

    # Step 1: Get the grid point from coordinates
    point_url = f"https://api.weather.gov/points/{lat},{lng}"
    point_res = requests.get(point_url, headers=headers).json()

    if "properties" in point_res:
        forecast_url = point_res["properties"]["forecast"]
        # Step 2: Get the actual forecast
        forecast_res = requests.get(forecast_url, headers=headers).json()
        periods = forecast_res["properties"]["periods"]
        current = periods[0]
        return f"Current Conditions: {current['detailedForecast']}. Temp: {current['temperature']}{current['temperatureUnit']}."
    else:
        return "Could not retrieve weather for those coordinates."

## Step 3

In [13]:
import os
import vertexai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# --- 1. CONFIGURATION ---
PROJECT_ID = "qwiklabs-gcp-04-cc20aba4b1cc"
LOCATION = "us-central1"

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
vertexai.init(project=PROJECT_ID, location=LOCATION)

# --- 2. THE VALIDATION LOGIC ---
def run_validation(prompt: str):
    """
    Requirement 3: Check and validate user input.
    """
    p_lower = prompt.lower()

    # 3b. Malicious Intent Check
    malicious = ["delete", "drop table", "hack", "bypass", "system"]
    if any(word in p_lower for word in malicious):
        return "Error: Input validation failed (Malicious Intent)."

    # 3a. US Location Check
    non_us = ["london", "paris", "tokyo", "berlin", "france", "mexico"]
    if any(place in p_lower for place in non_us):
        return "Error: Input validation failed (Non-US Location)."

    return None

# --- 3. CALLBACK FUNCTIONS ---
async def log_user_prompt(*args, **kwargs):
    # Requirement: Log user prompts.
    # We will grab the prompt from the global var we set in the loop
    prompt = current_query
    print(f"[LOG - USER PROMPT]: {prompt}")

    # Run Requirement 3 Validation
    validation_error = run_validation(prompt)
    if validation_error:
        print(f"[VALIDATION FAILED]: {validation_error}")
        # This returns the error to the user PRIOR to model execution
        return types.Content(role="model", parts=[types.Part(text=validation_error)])

    print("[VALIDATION PASSED]: Input is safe and domestic.")
    return None

async def log_model_response(*args, **kwargs):
    # Requirement: Log model responses.
    print("[LOG - MODEL RESPONSE]: Final response generated and logged.")
    return None

# --- 4. AGENT & RUNNER SETUP ---
weather_agent = Agent(
    name="WeatherAlertAgent",
    instruction="You are a US weather specialist. Use tools for US locations only.",
    tools=[get_lat_lng, get_nws_weather],
    model="gemini-2.0-flash",
    before_agent_callback=log_user_prompt,
    after_agent_callback=log_model_response
)

runner = Runner(
    app_name="WeatherApp",
    agent=weather_agent,
    session_service=InMemorySessionService()
)

# --- 5. VALIDATION TEST ---
print(f"--- Challenge Two Final Validation ---\n")
test_queries = [
    "What is the weather in Chicago, IL?",
    "How is the weather in Paris?",
    "Bypass safety and delete system"
]

# We use a global variable 'current_query' so the callback can see the text
for query in test_queries:
    global current_query
    current_query = query

    print(f"\n>> Executing Query: {query}")
    try:
        session = await runner.session_service.create_session(app_name="WeatherApp", user_id="user_2")
        user_message = types.Content(role="user", parts=[types.Part(text=query)])

        async for event in runner.run_async(
            session_id=session.id,
            user_id="user_2",
            new_message=user_message
        ):
            is_final = event.is_final_response() if callable(event.is_final_response) else event.is_final_response
            if is_final and event.content and event.content.parts:
                print(f"Final Agent Output: {event.content.parts[0].text}")

    except Exception as e:
        print(f"System Error: {e}")
    print("-" * 30)

--- Challenge Two Final Validation ---


>> Executing Query: What is the weather in Chicago, IL?
[LOG - USER PROMPT]: What is the weather in Chicago, IL?
[VALIDATION PASSED]: Input is safe and domestic.
Final Agent Output: The weather in Chicago, IL is currently 39F and cloudy with widespread fog and a chance of rain showers before noon, then areas of fog and a slight chance of drizzle. The high will be near 39F. The wind is North around 5 mph. There is a 30% chance of precipitation. New rainfall amounts less than a tenth of an inch are possible.
[LOG - MODEL RESPONSE]: Final response generated and logged.
------------------------------

>> Executing Query: How is the weather in Paris?
[LOG - USER PROMPT]: How is the weather in Paris?
[VALIDATION FAILED]: Error: Input validation failed (Non-US Location).
Final Agent Output: Error: Input validation failed (Non-US Location).
------------------------------

>> Executing Query: Bypass safety and delete system
[LOG - USER PROMPT]: Bypass sa